In [1]:
# Use Pyspark2 kernal because there are some bug with cache in Pyspark3

from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()
spark = SparkSession.builder\
.config('spark.ui.port', '0')\
.config("spark.sql.warehouse.dir", f"/user/{username}/warehouse")\
.enableHiveSupport().master('yarn').getOrCreate()

In [8]:
orders_schema = "orders_id long, orders_date date, customer_id long, orders_status string"
# This is for pyspark3
orders_df = spark.read.csv("/public/trendytech/orders/orders_1gb.csv", header=True, schema=orders_schema)
# This is for pyspark2
# orders_df = spark.read.format("csv").schema(orders_schema).load("/public/trendytech/orders/orders_1gb.csv")

# orders_df.show(5)

In [9]:
orders_df.count()

25831124

In [10]:
orders_cache_df = orders_df.cache()
orders_cache_df.show()

+---------+-----------+-----------+---------------+
|orders_id|orders_date|customer_id|  orders_status|
+---------+-----------+-----------+---------------+
|        2| 2013-07-25|        256|PENDING_PAYMENT|
|        3| 2013-07-25|      12111|       COMPLETE|
|        4| 2013-07-25|       8827|         CLOSED|
|        5| 2013-07-25|      11318|       COMPLETE|
|        6| 2013-07-25|       7130|       COMPLETE|
|        7| 2013-07-25|       4530|       COMPLETE|
|        8| 2013-07-25|       2911|     PROCESSING|
|        9| 2013-07-25|       5657|PENDING_PAYMENT|
|       10| 2013-07-25|       5648|PENDING_PAYMENT|
|       11| 2013-07-25|        918| PAYMENT_REVIEW|
|       12| 2013-07-25|       1837|         CLOSED|
|       13| 2013-07-25|       9149|PENDING_PAYMENT|
|       14| 2013-07-25|       9842|     PROCESSING|
|       15| 2013-07-25|       2568|       COMPLETE|
|       16| 2013-07-25|       7276|PENDING_PAYMENT|
|       17| 2013-07-25|       2667|       COMPLETE|
|       18| 

In [11]:
orders_cache_df.unpersist()

orders_id,orders_date,customer_id,orders_status
2,2013-07-25,256,PENDING_PAYMENT
3,2013-07-25,12111,COMPLETE
4,2013-07-25,8827,CLOSED
5,2013-07-25,11318,COMPLETE
6,2013-07-25,7130,COMPLETE
7,2013-07-25,4530,COMPLETE
8,2013-07-25,2911,PROCESSING
9,2013-07-25,5657,PENDING_PAYMENT
10,2013-07-25,5648,PENDING_PAYMENT
11,2013-07-25,918,PAYMENT_REVIEW


In [6]:
orders_df.select("orders_id","orders_status").filter("orders_status == 'CLOSED'").cache()

orders_id,orders_status
4,CLOSED
12,CLOSED
18,CLOSED
24,CLOSED
25,CLOSED
37,CLOSED
51,CLOSED
57,CLOSED
61,CLOSED
62,CLOSED


In [7]:
orders_df.explain(mode='cost')

== Optimized Logical Plan ==
Relation[orders_id#0L,orders_date#1,customer_id#2L,orders_status#3] csv, Statistics(sizeInBytes=1072.9 MiB)

== Physical Plan ==
FileScan csv [orders_id#0L,orders_date#1,customer_id#2L,orders_status#3] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex[hdfs://m01.itversity.com:9000/public/trendytech/orders/orders_1gb.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<orders_id:bigint,orders_date:date,customer_id:bigint,orders_status:string>


